# Examine output

use this notebook to see how effective the processing is for 2022.

In [5]:
import pandas as pd
import geopandas as gpd
from geosnap.io.util import _normalize_relation

df21 = gpd.read_parquet("2021_bg/acs_demographic_profile_2021_bg.parquet")
df22 = gpd.read_parquet("2022_bg/acs_demographic_profile_2022_bg.parquet")

vars_df = pd.read_csv("../geosnap/io/variables.csv")

needed = set()
for rel in vars_df["acs"].dropna():
    expr = _normalize_relation(rel)
    pieces = (
        expr.replace("+", ",")
        .replace("-", ",")
        .replace("(", "")
        .replace(")", "")
        .split(",")
    )
    for piece in pieces:
        piece = piece.strip()
        if piece:
            needed.add(piece)

present21 = set(df21.columns)
present22 = set(df22.columns)

missing21 = sorted(needed - present21)
missing22 = sorted(needed - present22)

newly_missing_in_2022 = sorted(set(missing22) - set(missing21))
fixed_in_2022 = sorted((needed - present21) - (needed - present22))

print("Needed ACS vars:", len(needed))
print("Missing in 2021:", len(missing21))
print("Missing in 2022:", len(missing22))
print("Newly missing in 2022:", len(newly_missing_in_2022))
print("Missing in 2021 but present in 2022:", len(fixed_in_2022))

print("\nFirst 100 newly missing in 2022:")
print(newly_missing_in_2022[:100])

Needed ACS vars: 260
Missing in 2021: 171
Missing in 2022: 260
Newly missing in 2022: 89
Missing in 2021 but present in 2022: 0

First 100 newly missing in 2022:
['B01001_003E', 'B01001_004E', 'B01001_005E', 'B01001_006E', 'B01001_018E', 'B01001_019E', 'B01001_020E', 'B01001_021E', 'B01001_022E', 'B01001_023E', 'B01001_024E', 'B01001_025E', 'B01001_027E', 'B01001_028E', 'B01001_029E', 'B01001_030E', 'B01001_042E', 'B01001_043E', 'B01001_044E', 'B01001_045E', 'B01001_046E', 'B01001_047E', 'B01001_048E', 'B01001_049E', 'B01003_001E', 'B02001_006E', 'B03002_003E', 'B03002_004E', 'B03002_005E', 'B03002_006E', 'B03002_007E', 'B03002_012E', 'B12001_001E', 'B12001_005E', 'B12001_007E', 'B12001_009E', 'B12001_010E', 'B12001_016E', 'B12001_018E', 'B12001_019E', 'B15002_001E', 'B15002_003E', 'B15002_004E', 'B15002_005E', 'B15002_006E', 'B15002_007E', 'B15002_008E', 'B15002_009E', 'B15002_010E', 'B15002_015E', 'B15002_016E', 'B15002_017E', 'B15002_018E', 'B15002_020E', 'B15002_021E', 'B15002_022E

# Are variables gone or just renamed?
per @knaaptime: Its possible these have different analogues/variable names in the new ACS but we'd need to dig more

let's go dig more

In [6]:
import requests

vars2021 = requests.get("https://api.census.gov/data/2021/acs/acs5/variables.json").json()["variables"]
vars2022 = requests.get("https://api.census.gov/data/2022/acs/acs5/variables.json").json()["variables"]

In [7]:
missing = newly_missing_in_2022

still_exist = []
gone = []

for var in missing:
    if var in vars2022:
        still_exist.append(var)
    else:
        gone.append(var)

print("Still exist in 2022:", len(still_exist))
print("Gone in 2022:", len(gone))

Still exist in 2022: 89
Gone in 2022: 0


So like, the variables still exist per the metadata (variables json provided by the ACS), but they are not present where I expect them (tiger product). First, let's ID all the variables by their census identifiers:

In [12]:
for var in still_exist:
    print(var, "->", vars2022[var]["label"])

B01001_003E -> Estimate!!Total:!!Male:!!Under 5 years
B01001_004E -> Estimate!!Total:!!Male:!!5 to 9 years
B01001_005E -> Estimate!!Total:!!Male:!!10 to 14 years
B01001_006E -> Estimate!!Total:!!Male:!!15 to 17 years
B01001_018E -> Estimate!!Total:!!Male:!!60 and 61 years
B01001_019E -> Estimate!!Total:!!Male:!!62 to 64 years
B01001_020E -> Estimate!!Total:!!Male:!!65 and 66 years
B01001_021E -> Estimate!!Total:!!Male:!!67 to 69 years
B01001_022E -> Estimate!!Total:!!Male:!!70 to 74 years
B01001_023E -> Estimate!!Total:!!Male:!!75 to 79 years
B01001_024E -> Estimate!!Total:!!Male:!!80 to 84 years
B01001_025E -> Estimate!!Total:!!Male:!!85 years and over
B01001_027E -> Estimate!!Total:!!Female:!!Under 5 years
B01001_028E -> Estimate!!Total:!!Female:!!5 to 9 years
B01001_029E -> Estimate!!Total:!!Female:!!10 to 14 years
B01001_030E -> Estimate!!Total:!!Female:!!15 to 17 years
B01001_042E -> Estimate!!Total:!!Female:!!60 and 61 years
B01001_043E -> Estimate!!Total:!!Female:!!62 to 64 year

# Group by census table
Now, let's group those into their repsective census tables to figure out what still needs to be missing

In [13]:
import re
from collections import defaultdict

In [14]:
def variable_to_table_group(var: str) -> str | None:
    """
    Convert an ACS variable like B01003_001E to its table/group name B01003.
    """
    m = re.match(r"^([A-Z0-9]+)_\d+[A-Z]$", var)
    if m:
        return m.group(1)
    return None


def group_variables_by_table(vars_list: list[str]) -> dict[str, list[str]]:
    groups = defaultdict(list)
    unparsed = []

    for var in sorted(set(vars_list)):
        group = variable_to_table_group(var)
        if group is None:
            unparsed.append(var)
        else:
            groups[group].append(var)

    if unparsed:
        print("Could not parse these variables:")
        for var in unparsed:
            print("  ", var)

    return dict(sorted(groups.items()))


groups = group_variables_by_table(newly_missing_in_2022)

print(f"Unique table groups: {len(groups)}")
for group, vars_ in groups.items():
    print(f"{group}: {len(vars_)} vars")

Unique table groups: 17
B01001: 24 vars
B01003: 1 vars
B02001: 1 vars
B03002: 6 vars
B12001: 8 vars
B15002: 25 vars
B17010: 4 vars
B19001: 1 vars
B19013: 1 vars
B19301: 1 vars
B21001: 1 vars
B25002: 3 vars
B25003: 3 vars
B25024: 7 vars
B25058: 1 vars
B25077: 1 vars
C24010: 1 vars


## Another way to inspect the tables/groups

In [ ]:
def describe_group(group: str, vars_meta: dict) -> pd.DataFrame:
    rows = []
    for var, meta in vars_meta.items():
        if var.startswith(f"{group}_"):
            rows.append(
                {
                    "variable": var,
                    "label": meta.get("label"),
                    "concept": meta.get("concept"),
                    "predicateType": meta.get("predicateType"),
                    "group": meta.get("group"),
                }
            )
    return pd.DataFrame(rows).sort_values("variable")

In [16]:
# Example:
describe_group("B01001", vars2022).head(20)

,variable,label,concept,predicateType,group
40,B01001_001E,Estimate!!Total:,Sex by Age,int,B01001
39,B01001_002E,Estimate!!Total:!!Male:,Sex by Age,int,B01001
42,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,Sex by Age,int,B01001
41,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,Sex by Age,int,B01001
45,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,Sex by Age,int,B01001
43,B01001_006E,Estimate!!Total:!!Male:!!15 to 17 years,Sex by Age,int,B01001
44,B01001_007E,Estimate!!Total:!!Male:!!18 and 19 years,Sex by Age,int,B01001
47,B01001_008E,Estimate!!Total:!!Male:!!20 years,Sex by Age,int,B01001
46,B01001_009E,Estimate!!Total:!!Male:!!21 years,Sex by Age,int,B01001
48,B01001_010E,Estimate!!Total:!!Male:!!22 to 24 years,Sex by Age,int,B01001
